# Heterogeneous mixing and transmission assumptions

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/13-mixing-and-transmission-types.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Four assumption combinations — density vs frequency transmission, crossed with
unstratified vs two-group stratified — should give **identical** prevalence
when the stratified models use homogeneous (or frequency-balanced) mixing and
split the seed with the groups.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    REMAINDER,
    Compartments,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Split,
    FlowModel,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

END = 40.0
POP = 1.0
SEED = 0.01
PARAMS = {
    "risk_per_contact": 0.5,
    "infectious_period": 4.0,
    "prop1": float(np.random.uniform()),
    "mixing_value": float(np.random.uniform()),
}
plan = SavePlan(requests={"comp": SaveRequest(Compartments())})


def prevalence_series(res, state, pmap):
    frame = res["comp"].select(state["infectious"]).to_pandas()
    return frame.sum(axis=1)


## Four combinations

Unstratified models still need a singleton mixing axis (`pop`). Stratified
models split the population and the seed with `prop1` / `REMAINDER`.

In [ ]:
def run_case(*, kind: str, stratified: bool) -> pd.Series:
    state = Property("state", ("susceptible", "infectious", "recovered"))
    if stratified:
        groups = Property("groups", ("group1", "group2"))
        pmap = PropertyMap.from_property(state).stratify(groups)
        epi = FlowModel(pmap)
        if kind == "density":
            K = np.ones((2, 2))
        else:
            m = PARAMS["mixing_value"]
            K = np.array([[m, 1.0 - m], [1.0 - m, m]])
        
        mixing = MixingMatrix(groups, K, normalize=("none" if kind == "density" else "rows"), check_reciprocal=False)
        foi = ForceOfInfection(
            "infection",
            infectious=state["infectious"],
            group_by=groups,
            kind=kind,
            contact_rate=Param("risk_per_contact"),
            mixing=mixing,
        )
        epi.add_flow(
            TransitionFlow(
                "infection",
                state["susceptible"],
                state["infectious"],
                foi,
            )
        )
        epi.add_flow(TransitionFlow(
            "recovery", state["infectious"], state["recovered"], 1.0 / Param("infectious_period")
        ))
        epi.set_initial_population(
            {
                state["susceptible"]: POP - SEED,
                state["infectious"]: SEED,
            },
            splits=(Split(groups, {"group1": Param("prop1"), "group2": REMAINDER}),),
        )
    else:
        pop = Property("pop", ("all",))
        pmap = PropertyMap.from_property(state).stratify(pop)
        epi = FlowModel(pmap)
        
        mixing = MixingMatrix(pop, [[1.0]], check_reciprocal=False)
        foi = ForceOfInfection(
            "infection",
            infectious=state["infectious"],
            group_by=pop,
            kind=kind,
            contact_rate=Param("risk_per_contact"),
            mixing=mixing,
        )
        epi.add_flow(
            TransitionFlow(
                "infection",
                state["susceptible"],
                state["infectious"],
                foi,
            )
        )
        epi.add_flow(TransitionFlow(
            "recovery", state["infectious"], state["recovered"], 1.0 / Param("infectious_period")
        ))
        epi.set_initial_population(
            {state["susceptible"]: POP - SEED, state["infectious"]: SEED}
        )
    cm = epi.compile()
    res = cm.run(PARAMS, t0=0.0, t1=END, dt=0.1, save=plan, solver="euler")
    return prevalence_series(res, state, pmap)


outputs = pd.DataFrame(
    {
        "dens_unstratified": run_case(kind="density", stratified=False),
        "dens_stratified": run_case(kind="density", stratified=True),
        "freq_unstratified": run_case(kind="frequency", stratified=False),
        "freq_stratified": run_case(kind="frequency", stratified=True),
    }
)
differences = outputs.min(axis=1) - outputs.max(axis=1)
assert all(abs(differences) < 1e-5), "There's a discrepancy"
outputs.plot(labels={"index": "time", "value": "prevalence"})


Phew — no interesting dynamics yet, but the four curves overlie. That is the
starting point for heterogeneous mixing in later chapters.